## Movie Recommender Comparison

In [1]:
import pandas as pd
import numpy as np

ratings = pd.read_csv("dataset/ratings.csv")  # userId, movieId, rating
movies = pd.read_csv("dataset/movies.csv")    # movieId, title, genres

print(ratings.head())
print(movies.head())

   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  


## Leave-One-Out

In [2]:
def leave_one_out_split(ratings_df):
    # Shuffle to avoid bias if no timestamp
    ratings_df = ratings_df.sample(frac=1, random_state=42)

    test = ratings_df.groupby("userId").head(1)
    train = ratings_df.drop(test.index)

    return train, test


train_df, test_df = leave_one_out_split(ratings)

print("Train size:", len(train_df))
print("Test size:", len(test_df))

Train size: 100226
Test size: 610


## Build popularity baseline

In [3]:
def build_popularity_baseline(train_df, min_ratings=5):
    movie_stats = (
        train_df
        .groupby("movieId")
        .agg(
            avg_rating=("rating", "mean"),
            rating_count=("rating", "count")
        )
        .reset_index()
    )

    #filter low-count
    movie_stats = movie_stats[movie_stats["rating_count"] >= min_ratings]

    #sort by rating then count
    movie_stats = movie_stats.sort_values(
        by=["avg_rating", "rating_count"],
        ascending=False
    )

    return movie_stats


popularity_df = build_popularity_baseline(train_df)

popularity_df.head(10)

,movieId,avg_rating,rating_count
4385,6460,4.900000,5
9588,177593,4.750000,8
5753,31364,4.700000,5
1662,2239,4.666667,6
1424,1949,4.600000,5
3200,4334,4.600000,5
796,1041,4.590909,11
8274,106642,4.571429,7
2576,3451,4.545455,11
881,1178,4.541667,12


In [4]:
def recommend_popular_movies(user_id, train_df, popularity_df, n=10):
    seen_movies = set(train_df[train_df["userId"] == user_id]["movieId"])

    recs = popularity_df[
        ~popularity_df["movieId"].isin(seen_movies)
    ].head(n)

    return recs["movieId"].tolist()


#example
user_id = ratings["userId"].iloc[0]
rec_ids = recommend_popular_movies(user_id, train_df, popularity_df, n=10)

print(rec_ids)

[6460, 177593, 31364, 2239, 1949, 4334, 1041, 106642, 3451, 1178]


In [5]:
def show_recommendations(movie_ids, movies_df):
    return movies_df[movies_df["movieId"].isin(movie_ids)][["movieId", "title"]]


show_recommendations(rec_ids, movies)

,movieId,title
796,1041,Secrets & Lies (1996)
883,1178,Paths of Glory (1957)
1426,1949,"Man for All Seasons, A (1966)"
1664,2239,Swept Away (Travolti da un insolito destino ne...
2582,3451,Guess Who's Coming to Dinner (1967)
3210,4334,Yi Yi (2000)
4396,6460,"Trial, The (Procès, Le) (1962)"
5773,31364,Memories of Murder (Salinui chueok) (2003)
8301,106642,"Day of the Doctor, The (2013)"
9618,177593,"Three Billboards Outside Ebbing, Missouri (2017)"


In [6]:
def precision_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & set(relevant))
    return hits / k

def recall_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & set(relevant))
    return hits / len(relevant) if len(relevant) > 0 else 0

def hit_rate_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    return int(len(set(recommended_k) & set(relevant)) > 0)

def evaluate_baseline(train_df, test_df, popularity_df, k=10):
    precisions = []
    recalls = []
    hits = []

    for _, row in test_df.iterrows():
        user_id = row["userId"]
        test_movie = row["movieId"]

        recs = recommend_popular_movies(user_id, train_df, popularity_df, n=k)
        relevant = [test_movie]

        precisions.append(precision_at_k(recs, relevant, k))
        recalls.append(recall_at_k(recs, relevant, k))
        hits.append(hit_rate_at_k(recs, relevant, k))

    return {
        "Model": "Popularity Baseline",
        f"Precision@{k}": np.mean(precisions),
        f"Recall@{k}": np.mean(recalls),
        f"HitRate@{k}": np.mean(hits)
    }


results = evaluate_baseline(train_df, test_df, popularity_df, k=10)

results_df = pd.DataFrame([results])
print(results_df)

                 Model  Precision@10  Recall@10  HitRate@10
0  Popularity Baseline      0.000164   0.001639    0.001639


## kNN

In [7]:
from sklearn.neighbors import NearestNeighbors

def build_user_item_matrix(train_df):
    """
    Rows = users
    Columns = movies
    Values = ratings
    Missing ratings filled with 0 for cosine-based kNN.
    """
    user_item_matrix = train_df.pivot_table(
        index="userId",
        columns="movieId",
        values="rating"
    ).fillna(0)

    return user_item_matrix


user_item_matrix = build_user_item_matrix(train_df)

print("User-item matrix shape:", user_item_matrix.shape)
user_item_matrix.head()

User-item matrix shape: (610, 9712)


movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,0.0,4.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
def fit_item_knn_model(user_item_matrix, k_neighbors=20):
    """
    Fits an item-item kNN model using cosine distance.

    sklearn returns distances, not similarities:
    cosine similarity = 1 - cosine distance
    """
    item_user_matrix = user_item_matrix.T

    knn_model = NearestNeighbors(
        metric="cosine",
        algorithm="brute",
        n_neighbors=k_neighbors + 1  # +1 because item is nearest to itself
    )

    knn_model.fit(item_user_matrix)

    return knn_model, item_user_matrix


k_neighbors = 20

knn_model, item_user_matrix = fit_item_knn_model(
    user_item_matrix,
    k_neighbors=k_neighbors
)

print("Item-user matrix shape:", item_user_matrix.shape)

Item-user matrix shape: (9712, 610)


In [9]:
def build_item_neighbors(knn_model, item_user_matrix, k_neighbors=20):
    """
    Builds a dictionary:
    item_id -> list of (neighbor_item_id, similarity)

    The item itself is removed from its neighbor list.
    """
    distances, indices = knn_model.kneighbors(item_user_matrix)

    item_ids = item_user_matrix.index.to_list()

    item_neighbors = {}

    for item_pos, item_id in enumerate(item_ids):
        neighbors = []

        for dist, neighbor_pos in zip(distances[item_pos], indices[item_pos]):
            neighbor_id = item_ids[neighbor_pos]

            # Skip the item itself
            if neighbor_id == item_id:
                continue

            similarity = 1 - dist

            # Keep only positive similarities
            if similarity > 0:
                neighbors.append((neighbor_id, similarity))

        item_neighbors[item_id] = neighbors[:k_neighbors]

    return item_neighbors


item_neighbors = build_item_neighbors(
    knn_model,
    item_user_matrix,
    k_neighbors=k_neighbors
)

# Example: show neighbors for first movie in matrix
first_movie_id = item_user_matrix.index[0]
item_neighbors[first_movie_id][:5]

[(480, 0.5724462643532684),
 (3114, 0.571680394218728),
 (780, 0.5692411022872375),
 (260, 0.5549127608019504),
 (356, 0.5494405001390368)]

In [10]:
def score_user_items_knn(user_id, user_item_matrix, item_neighbors):
    """
    Scores unseen movies for a user using item-item kNN.

    Logic:
    - Look at movies the user has rated
    - For each rated movie, get its nearest neighbors
    - Add weighted scores to neighboring unseen movies
    - Higher score = stronger recommendation
    """
    if user_id not in user_item_matrix.index:
        return pd.Series(dtype=float)

    user_ratings = user_item_matrix.loc[user_id]

    rated_items = user_ratings[user_ratings > 0]
    seen_items = set(rated_items.index)

    scores = {}
    sim_sums = {}

    for rated_item_id, rating in rated_items.items():
        neighbors = item_neighbors.get(rated_item_id, [])

        for neighbor_id, similarity in neighbors:
            # Do not recommend movies the user already rated
            if neighbor_id in seen_items:
                continue

            scores[neighbor_id] = scores.get(neighbor_id, 0) + similarity * rating
            sim_sums[neighbor_id] = sim_sums.get(neighbor_id, 0) + similarity

    # Normalize weighted scores by similarity sum
    normalized_scores = {}

    for item_id in scores:
        if sim_sums[item_id] > 0:
            normalized_scores[item_id] = scores[item_id] / sim_sums[item_id]

    return pd.Series(normalized_scores).sort_values(ascending=False)

In [11]:
def recommend_item_knn(user_id, user_item_matrix, item_neighbors, n=10):
    """
    Returns Top-N movie IDs for one user.
    """
    scores = score_user_items_knn(
        user_id=user_id,
        user_item_matrix=user_item_matrix,
        item_neighbors=item_neighbors
    )

    return scores.head(n).index.tolist()


# Example recommendation
example_user = ratings["userId"].iloc[0]

knn_rec_ids = recommend_item_knn(
    user_id=example_user,
    user_item_matrix=user_item_matrix,
    item_neighbors=item_neighbors,
    n=10
)

print(knn_rec_ids)

[911, 6341, 7143, 6378, 2816, 5267, 4011, 2599, 3359, 6337]


In [12]:
def show_recommendations(movie_ids, movies_df):
    """
    Converts movie IDs into movie titles.
    """
    return movies_df[
        movies_df["movieId"].isin(movie_ids)
    ][["movieId", "title", "genres"]]


show_recommendations(knn_rec_ids, movies)

,movieId,title,genres
693,911,Charade (1963),Comedy|Crime|Mystery|Romance|Thriller
1960,2599,Election (1999),Comedy
2119,2816,Iron Eagle II (1988),Action|War
2511,3359,Breaking Away (1979),Comedy|Drama
2996,4011,Snatch (2000),Comedy|Crime|Thriller
3771,5267,"Rookie, The (2002)",Drama
4338,6337,Owning Mahowny (2003),Crime|Drama|Thriller
4341,6341,"Shape of Things, The (2003)",Drama
4361,6378,"Italian Job, The (2003)",Action|Crime
4795,7143,"Last Samurai, The (2003)",Action|Adventure|Drama|War


In [13]:
def precision_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & set(relevant))
    return hits / k


def recall_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & set(relevant))
    return hits / len(relevant) if len(relevant) > 0 else 0


def hit_rate_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    return int(len(set(recommended_k) & set(relevant)) > 0)

def evaluate_item_knn(test_df, user_item_matrix, item_neighbors, top_n=10):
    precisions = []
    recalls = []
    hits = []
    skipped_users = 0

    for _, row in test_df.iterrows():
        user_id = row["userId"]
        test_movie = row["movieId"]

        if user_id not in user_item_matrix.index:
            skipped_users += 1
            continue

        recs = recommend_item_knn(
            user_id=user_id,
            user_item_matrix=user_item_matrix,
            item_neighbors=item_neighbors,
            n=top_n
        )

        relevant = [test_movie]

        precisions.append(precision_at_k(recs, relevant, top_n))
        recalls.append(recall_at_k(recs, relevant, top_n))
        hits.append(hit_rate_at_k(recs, relevant, top_n))

    return {
        "Model": f"Item-Item kNN (k={k_neighbors})",
        f"Precision@{top_n}": np.mean(precisions),
        f"Recall@{top_n}": np.mean(recalls),
        f"HitRate@{top_n}": np.mean(hits),
        "Skipped Users": skipped_users
    }


knn_results = evaluate_item_knn(
    test_df=test_df,
    user_item_matrix=user_item_matrix,
    item_neighbors=item_neighbors,
    top_n=10
)

print(knn_results)

{'Model': 'Item-Item kNN (k=20)', 'Precision@10': 0.000819672131147541, 'Recall@10': 0.00819672131147541, 'HitRate@10': 0.00819672131147541, 'Skipped Users': 0}


In [14]:
knn_results_df = pd.DataFrame([knn_results])

print(knn_results_df)

                  Model  Precision@10  Recall@10  HitRate@10  Skipped Users
0  Item-Item kNN (k=20)       0.00082   0.008197    0.008197              0


In [15]:
combined_results_df = pd.concat(
    [
        results_df,
        knn_results_df.drop(columns=["Skipped Users"], errors="ignore")
    ],
    ignore_index=True
)

print(combined_results_df)

                  Model  Precision@10  Recall@10  HitRate@10
0   Popularity Baseline      0.000164   0.001639    0.001639
1  Item-Item kNN (k=20)      0.000820   0.008197    0.008197


In [16]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split

In [17]:
reader = Reader(rating_scale=(1, 5))

surprise_data = Dataset.load_from_df(
    ratings[["userId", "movieId", "rating"]],
    reader
)

trainset = surprise_data.build_full_trainset()

svd_model = SVD(
    n_factors=50,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

svd_model.fit(trainset)

In [18]:
def recommend_svd(user_id, ratings_df, movies_df, model, n=10):
    """
    Recommend Top-N unseen movies for a user using trained SVD model.
    """
    all_movie_ids = set(movies_df["movieId"].unique())

    seen_movies = set(
        ratings_df[ratings_df["userId"] == user_id]["movieId"]
    )

    unseen_movies = list(all_movie_ids - seen_movies)

    predictions = []

    for movie_id in unseen_movies:
        predicted_rating = model.predict(user_id, movie_id).est
        predictions.append((movie_id, predicted_rating))

    predictions.sort(key=lambda x: x[1], reverse=True)

    top_movie_ids = [movie_id for movie_id, score in predictions[:n]]

    return top_movie_ids

In [19]:
example_user = ratings["userId"].iloc[0]

svd_rec_ids = recommend_svd(
    user_id=example_user,
    ratings_df=train_df,
    movies_df=movies,
    model=svd_model,
    n=10
)

movies[movies["movieId"].isin(svd_rec_ids)][["movieId", "title", "genres"]]

,movieId,title,genres
277,318,"Shawshank Redemption, The (1994)",Crime|Drama
602,750,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War
659,858,"Godfather, The (1972)",Crime|Drama
681,899,Singin' in the Rain (1952),Comedy|Musical|Romance
686,904,Rear Window (1954),Mystery|Thriller
694,912,Casablanca (1942),Drama|Romance
841,1104,"Streetcar Named Desire, A (1951)",Drama
903,1201,"Good, the Bad and the Ugly, The (Buono, il bru...",Action|Adventure|Western
906,1204,Lawrence of Arabia (1962),Adventure|Drama|War
922,1221,"Godfather: Part II, The (1974)",Crime|Drama


In [20]:
def precision_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & set(relevant))
    return hits / k


def recall_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & set(relevant))
    return hits / len(relevant) if len(relevant) > 0 else 0


def hit_rate_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    return int(len(set(recommended_k) & set(relevant)) > 0)

In [21]:
def evaluate_svd(test_df, train_df, movies_df, model, top_n=10):
    precisions = []
    recalls = []
    hits = []

    for _, row in test_df.iterrows():
        user_id = row["userId"]
        test_movie = row["movieId"]

        recs = recommend_svd(
            user_id=user_id,
            ratings_df=train_df,
            movies_df=movies_df,
            model=model,
            n=top_n
        )

        relevant = [test_movie]

        precisions.append(precision_at_k(recs, relevant, top_n))
        recalls.append(recall_at_k(recs, relevant, top_n))
        hits.append(hit_rate_at_k(recs, relevant, top_n))

    return {
        "Model": "SVD (50 factors)",
        f"Precision@{top_n}": np.mean(precisions),
        f"Recall@{top_n}": np.mean(recalls),
        f"HitRate@{top_n}": np.mean(hits)
    }


svd_results = evaluate_svd(
    test_df=test_df,
    train_df=train_df,
    movies_df=movies,
    model=svd_model,
    top_n=10
)

svd_results

{'Model': 'SVD (50 factors)',
 'Precision@10': 0.0034426229508196723,
 'Recall@10': 0.03442622950819672,
 'HitRate@10': 0.03442622950819672}

In [22]:
svd_results_df = pd.DataFrame([svd_results])

combined_results_df = pd.concat(
    [
        results_df,
        knn_results_df.drop(columns=["Skipped Users"], errors="ignore"),
        svd_results_df
    ],
    ignore_index=True
)

combined_results_df

,Model,Precision@10,Recall@10,HitRate@10
0,Popularity Baseline,0.000164,0.001639,0.001639
1,Item-Item kNN (k=20),0.000820,0.008197,0.008197
2,SVD (50 factors),0.003443,0.034426,0.034426


In [23]:
import random

random.seed(42)
np.random.seed(42)

def get_sampled_candidates(user_id, test_movie, train_df, movies_df, n_neg=100):
    """
    Creates candidate set:
    - 1 positive item: held-out test movie
    - n_neg negative items: random unseen movies
    """
    all_movies = set(movies_df["movieId"].unique())

    seen_movies = set(
        train_df[train_df["userId"] == user_id]["movieId"]
    )

    unseen_movies = list(all_movies - seen_movies - {test_movie})

    n_sample = min(n_neg, len(unseen_movies))
    negative_samples = random.sample(unseen_movies, n_sample)

    candidates = negative_samples + [test_movie]

    return candidates

In [24]:
def precision_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & set(relevant))
    return hits / k


def recall_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & set(relevant))
    return hits / len(relevant) if len(relevant) > 0 else 0


def hit_rate_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    return int(len(set(recommended_k) & set(relevant)) > 0)

In [25]:
def recommend_popularity_sampled(user_id, train_df, popularity_df, candidates, n=10):
    """
    Popularity baseline over sampled candidate set only.
    """
    candidate_df = popularity_df[
        popularity_df["movieId"].isin(candidates)
    ].copy()

    recs = candidate_df.sort_values(
        by=["avg_rating", "rating_count"],
        ascending=False
    ).head(n)

    return recs["movieId"].tolist()


def evaluate_popularity_sampled(test_df, train_df, movies_df, popularity_df, top_n=10, n_neg=100):
    precisions = []
    recalls = []
    hits = []

    for _, row in test_df.iterrows():
        user_id = row["userId"]
        test_movie = row["movieId"]

        candidates = get_sampled_candidates(
            user_id=user_id,
            test_movie=test_movie,
            train_df=train_df,
            movies_df=movies_df,
            n_neg=n_neg
        )

        recs = recommend_popularity_sampled(
            user_id=user_id,
            train_df=train_df,
            popularity_df=popularity_df,
            candidates=candidates,
            n=top_n
        )

        relevant = [test_movie]

        precisions.append(precision_at_k(recs, relevant, top_n))
        recalls.append(recall_at_k(recs, relevant, top_n))
        hits.append(hit_rate_at_k(recs, relevant, top_n))

    return {
        "Model": "Popularity Baseline",
        f"Precision@{top_n}": np.mean(precisions),
        f"Recall@{top_n}": np.mean(recalls),
        f"HitRate@{top_n}": np.mean(hits),
        "Evaluation": f"Sampled Candidates ({n_neg} negatives)"
    }


popularity_sampled_results = evaluate_popularity_sampled(
    test_df=test_df,
    train_df=train_df,
    movies_df=movies,
    popularity_df=popularity_df,
    top_n=10,
    n_neg=100
)

popularity_sampled_results

{'Model': 'Popularity Baseline',
 'Precision@10': 0.04114754098360656,
 'Recall@10': 0.41147540983606556,
 'HitRate@10': 0.41147540983606556,
 'Evaluation': 'Sampled Candidates (100 negatives)'}

In [26]:
def recommend_item_knn_sampled(user_id, user_item_matrix, item_neighbors, candidates, n=10):
    """
    Item-item kNN recommendations restricted to sampled candidate set.
    """
    scores = score_user_items_knn(
        user_id=user_id,
        user_item_matrix=user_item_matrix,
        item_neighbors=item_neighbors
    )

    candidate_scores = scores[
        scores.index.isin(candidates)
    ]

    return candidate_scores.head(n).index.tolist()


def evaluate_item_knn_sampled(test_df, train_df, movies_df, user_item_matrix, item_neighbors, top_n=10, n_neg=100):
    precisions = []
    recalls = []
    hits = []
    skipped_users = 0

    for _, row in test_df.iterrows():
        user_id = row["userId"]
        test_movie = row["movieId"]

        if user_id not in user_item_matrix.index:
            skipped_users += 1
            continue

        candidates = get_sampled_candidates(
            user_id=user_id,
            test_movie=test_movie,
            train_df=train_df,
            movies_df=movies_df,
            n_neg=n_neg
        )

        recs = recommend_item_knn_sampled(
            user_id=user_id,
            user_item_matrix=user_item_matrix,
            item_neighbors=item_neighbors,
            candidates=candidates,
            n=top_n
        )

        relevant = [test_movie]

        precisions.append(precision_at_k(recs, relevant, top_n))
        recalls.append(recall_at_k(recs, relevant, top_n))
        hits.append(hit_rate_at_k(recs, relevant, top_n))

    return {
        "Model": "Item-Item kNN (k=20)",
        f"Precision@{top_n}": np.mean(precisions),
        f"Recall@{top_n}": np.mean(recalls),
        f"HitRate@{top_n}": np.mean(hits),
        "Skipped Users": skipped_users,
        "Evaluation": f"Sampled Candidates ({n_neg} negatives)"
    }


knn_sampled_results = evaluate_item_knn_sampled(
    test_df=test_df,
    train_df=train_df,
    movies_df=movies,
    user_item_matrix=user_item_matrix,
    item_neighbors=item_neighbors,
    top_n=10,
    n_neg=100
)

knn_sampled_results

{'Model': 'Item-Item kNN (k=20)',
 'Precision@10': 0.06245901639344261,
 'Recall@10': 0.6245901639344262,
 'HitRate@10': 0.6245901639344262,
 'Skipped Users': 0,
 'Evaluation': 'Sampled Candidates (100 negatives)'}

In [27]:
def recommend_svd_sampled(user_id, model, candidates, n=10):
    """
    SVD recommendations restricted to sampled candidate set.
    """
    predictions = []

    for movie_id in candidates:
        pred_rating = model.predict(user_id, movie_id).est
        predictions.append((movie_id, pred_rating))

    predictions.sort(key=lambda x: x[1], reverse=True)

    return [movie_id for movie_id, _ in predictions[:n]]


def evaluate_svd_sampled(test_df, train_df, movies_df, model, top_n=10, n_neg=100):
    precisions = []
    recalls = []
    hits = []

    for _, row in test_df.iterrows():
        user_id = row["userId"]
        test_movie = row["movieId"]

        candidates = get_sampled_candidates(
            user_id=user_id,
            test_movie=test_movie,
            train_df=train_df,
            movies_df=movies_df,
            n_neg=n_neg
        )

        recs = recommend_svd_sampled(
            user_id=user_id,
            model=model,
            candidates=candidates,
            n=top_n
        )

        relevant = [test_movie]

        precisions.append(precision_at_k(recs, relevant, top_n))
        recalls.append(recall_at_k(recs, relevant, top_n))
        hits.append(hit_rate_at_k(recs, relevant, top_n))

    return {
        "Model": "SVD (50 factors)",
        f"Precision@{top_n}": np.mean(precisions),
        f"Recall@{top_n}": np.mean(recalls),
        f"HitRate@{top_n}": np.mean(hits),
        "Evaluation": f"Sampled Candidates ({n_neg} negatives)"
    }


svd_sampled_results = evaluate_svd_sampled(
    test_df=test_df,
    train_df=train_df,
    movies_df=movies,
    model=svd_model,
    top_n=10,
    n_neg=100
)

svd_sampled_results

{'Model': 'SVD (50 factors)',
 'Precision@10': 0.04327868852459016,
 'Recall@10': 0.43278688524590164,
 'HitRate@10': 0.43278688524590164,
 'Evaluation': 'Sampled Candidates (100 negatives)'}

In [28]:
sampled_results_df = pd.DataFrame([
    popularity_sampled_results,
    knn_sampled_results,
    svd_sampled_results
])

sampled_results_df = sampled_results_df.drop(
    columns=["Skipped Users"],
    errors="ignore"
)

sampled_results_df

,Model,Precision@10,Recall@10,HitRate@10,Evaluation
0,Popularity Baseline,0.041148,0.411475,0.411475,Sampled Candidates (100 negatives)
1,Item-Item kNN (k=20),0.062459,0.624590,0.624590,Sampled Candidates (100 negatives)
2,SVD (50 factors),0.043279,0.432787,0.432787,Sampled Candidates (100 negatives)
